# rmul-scalar-tensor-mix — ex1: implement __mul__ and __rmul__ so both 2*t and t*2 work

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rmul-scalar-tensor-mix`. Running the final beacon cell reports progress against the `PyTorch: __rmul__ scalar/tensor mix` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: __rmul__ scalar/tensor mix` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rmul-scalar-tensor-mix`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rmul-scalar-tensor-mix"
DD_SUBTOPIC = "PyTorch: __rmul__ scalar/tensor mix"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PyTorch: `__rmul__` scalar/tensor mix — quick refresher

When you write `3 * tensor`, Python first tries `int.__mul__(3, tensor)`. `int` doesn't know what a `Tensor` is, so it returns `NotImplemented`. Python then calls `tensor.__rmul__(3)` — the *reflected* multiply. That's how scalar-on-the-left works:

```python
class MiniTensor:
    def __mul__(self, other):    return self._mul(other)   # tensor * x
    def __rmul__(self, other):   return self._mul(other)   # x * tensor (x doesn't know us)
```

**Both directions MUST exist.** `tensor * 3` dispatches to `__mul__`. `3 * tensor` dispatches to `__rmul__`. If you only implement `__mul__`, the scalar-on-the-left form raises `TypeError`.

**For multiplication, the two are usually symmetric.** `a * b == b * a` for scalar-tensor mixes, so `__rmul__` can simply delegate to `__mul__`. For NON-commutative ops (`__matmul__` / `__rmatmul__`, `__sub__` / `__rsub__`), the reflected version must SWAP the operand order: `__rsub__(self, other)` returns `other - self`, not `self - other`.

**Why ARENA's MiniTensor needs both.** Tests will write expressions like `2 * x` (scalar literal first) and `x * 2` (tensor first) interchangeably. The framework cannot assume one ordering. Implement both to make every test pass.

**The same applies to `__add__`/`__radd__`, `__sub__`/`__rsub__`, `__truediv__`/`__rtruediv__`, etc.** All the binary numeric dunders come in `__op__` + `__rop__` pairs.

### Exercise 1 — implement __mul__ and __rmul__ so both 2*t and t*2 work

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `__mul__` / `__rmul__` dunder pair to a custom tensor wrapper so scalar*tensor and tensor*scalar both dispatch correctly, exploiting the commutativity of multiplication.
> Keywords: dunder, rmul, reflected-op, scalar-tensor
> ```

**KCs targeted:** `rmul-implements-reflected-multiply`, `mul-rmul-symmetry-for-commutative-op`

Complete the `Wrapper` class below by implementing `__mul__` and `__rmul__` so that both `2 * w` (scalar on the left) and `w * 2` (scalar on the right) return a new `Wrapper` containing `2 * w.data`.

Inputs:
- `self.data` is a `torch.Tensor` stored on the wrapper.
- The other operand is a Python scalar (int or float).

Output: a NEW `Wrapper` whose `.data` is the elementwise product. Do not mutate `self`.

Constraints:
- BOTH `__mul__` and `__rmul__` must be implemented.
- `__rmul__` may delegate to `__mul__` (multiplication is commutative for scalar/tensor mixes).
- Return a `Wrapper`, NOT a raw `Tensor`.

In [ ]:
class Wrapper:
    def __init__(self, data):
        self.data = data

    def __mul__(self, other):
        raise NotImplementedError()

    def __rmul__(self, other):
        raise NotImplementedError()

    def __repr__(self):
        return f'Wrapper({self.data.tolist()})'


def _test_ex1():
    # === tensor * scalar (left-multiply) ===
    w = Wrapper(t.tensor([1.0, 2.0, 3.0]))
    out = w * 2
    assert isinstance(out, Wrapper), f'expected Wrapper, got {type(out).__name__}'
    assert t.allclose(out.data, t.tensor([2.0, 4.0, 6.0])), f'got {out.data}'

    # === scalar * tensor (right-multiply, dispatches to __rmul__) ===
    out = 2 * w
    assert isinstance(out, Wrapper), f'expected Wrapper from scalar*w, got {type(out).__name__}'
    assert t.allclose(out.data, t.tensor([2.0, 4.0, 6.0])), f'got {out.data}'

    # === Both directions produce the same result (commutativity) ===
    for scalar in [-1.0, 0.0, 0.5, 3, 7.5]:
        left = scalar * w
        right = w * scalar
        assert t.allclose(left.data, right.data), (
            f'asymmetric for scalar={scalar}: left={left.data} right={right.data}'
        )
        assert isinstance(left, Wrapper)
        assert isinstance(right, Wrapper)

    # === Does NOT mutate self ===
    original = t.tensor([1.0, 2.0, 3.0])
    w = Wrapper(original.clone())
    _ = 5 * w
    assert t.allclose(w.data, original), 'rmul should not mutate self'
    _ = w * 5
    assert t.allclose(w.data, original), 'mul should not mutate self'

    # === Confirm __rmul__ is actually being called (proves the dispatch path) ===
    # Python only calls __rmul__ when the left operand's __mul__ returned NotImplemented
    # or doesn't know how to handle the right operand. int.__mul__(2, Wrapper) returns
    # NotImplemented, so Python falls back to Wrapper.__rmul__(2). If __rmul__ were
    # missing, 2 * w would raise TypeError.
    w = Wrapper(t.tensor([10.0]))
    try:
        result = 3 * w
    except TypeError as e:
        raise AssertionError(
            f'scalar * Wrapper raised TypeError — did you forget __rmul__? ({e})'
        ) from None
    assert t.allclose(result.data, t.tensor([30.0]))

    # === Larger tensor smoke test ===
    big = t.randn(64, generator=t.Generator().manual_seed(0))
    w = Wrapper(big.clone())
    out_left = 1.5 * w
    out_right = w * 1.5
    assert t.allclose(out_left.data, out_right.data)
    assert t.allclose(out_left.data, big * 1.5)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class Wrapper:
    def __init__(self, data):
        self.data = data

    def __mul__(self, other):
        return Wrapper(self.data * other)

    def __rmul__(self, other):
        # Multiplication is commutative for scalar/tensor mixes, so we can
        # safely delegate to __mul__. For NON-commutative ops (like __sub__),
        # __rsub__ would need to compute other - self.data instead.
        return self.__mul__(other)

    def __repr__(self):
        return f'Wrapper({self.data.tolist()})'
```

**`__rmul__` makes scalar-on-the-left work.** Python tries `int.__mul__(3, wrapper)` first; `int` returns `NotImplemented`; Python then calls `wrapper.__rmul__(3)`. Without `__rmul__`, `3 * wrapper` raises `TypeError`.

**Delegation is fine for commutative ops.** `self.__mul__(other)` works because `a * b == b * a` for scalar/tensor multiplication. For `__rsub__`, you'd need `return Wrapper(other - self.data)` — the operand swap is essential.

**The same pair applies to `__add__`/`__radd__`, `__truediv__`/`__rtruediv__`, `__matmul__`/`__rmatmul__`, etc.** Every binary numeric dunder has a reflected counterpart. ARENA's MiniTensor implements ALL of them.

**Return a new `Wrapper`, never a raw `Tensor`.** Otherwise chains like `(2 * w) * 3` break — the intermediate would lose its wrapper identity.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()